### Structured output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x13901f8c0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x1391046e0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from pydantic import BaseModel,Field
class Movie(BaseModel):
    title:str=Field(description="The tile of the Movie")
    year:int=Field(description="This year the movie is released")
    director:str=Field(description='The Director of the movie')
    raring:float=Field(descriptio="The movie rating out of 10")

/var/folders/8t/bggqp3215xb0stby2r99gqcc0000gn/T/ipykernel_30005/183377994.py:6: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'descriptio'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  raring:float=Field(descriptio="The movie rating out of 10")


In [5]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x13901f8c0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x1391046e0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description

In [7]:
model.invoke("Provide details about the movie Incepton")

AIMessage(content='<think>\nOkay, so I need to provide details about the movie Inception. Let me start by recalling what I know about it. Inception is directed by Christopher Nolan, right? It came out in 2010, I think. The main actor is Leonardo DiCaprio, and the movie is about dreams and entering people\'s minds. The title itself, "Inception," refers to the act of planting an idea into someone\'s subconscious, which is a key plot point.\n\nThe story probably involves some kind of heist or mission where the characters enter dreams to steal information. The protagonist, Dom Cobb, is a thief who can enter people\'s dreams. He\'s offered a chance to erase his criminal past by performing the reverse of his usual job—planting an idea instead of stealing it. There\'s a lot of action and visual effects, especially with the use of the "kick" concept, which is a method to wake up from a dream.\n\nI remember there\'s a scene where a city folds in on itself, which was a big visual effect. The cas

In [9]:
response=model_with_structure.invoke("Provide details about the movie Incepton")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', raring=8.8)

### Message output alongside parsed structure

In [10]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for details about the movie Inception. Let me check the tools provided. There's a Movie function that requires title, year, director, and rating. I need to fill those in. I remember Inception was directed by Christopher Nolan. It came out in 2010. The rating is probably around 8.8 on IMDb. Let me confirm the exact year and director. Yep, 2010 and Christopher Nolan. The rating is 8.8. So I'll structure the tool call with those parameters.\n", 'tool_calls': [{'id': 'ed64qkban', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 163, 'prompt_tokens': 231, 'total_tokens': 394, 'completion_time': 0.913822494, 'completion_tokens_details': {'reasoning_tokens': 115}, 'prompt_time': 0.01155859, 'prompt_tokens_details': None, 'queue_time': 0.140355409

### Nested Structure

In [11]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Jack'), Actor(name='Ken Watanabe', role='Saito')], genres=['Action', 'Science Fiction', 'Thriller'], budget=160.0)

### TypedDict
TypedDict provides a simpler alternative using python's bilt-in-type ideal when you don't need runtimes validation.

In [13]:
from typing_extensions import TypedDict, Annotated

class MovieDetails(TypedDict):
    """A movie with details"""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]
    
model_withtypedict= model.with_structured_output(MovieDetails)
response = model_withtypedict.invoke("Please provide the details of the movie avengers")
response 

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

In [14]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avenger")
response

{'budget': 200000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Iron Man'},
  {'name': 'Chris Evans', 'role': 'Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Hawkeye'}],
 'genres': ['Action', 'Science Fiction'],
 'title': 'Avenger',
 'year': 2012}

In [16]:
model.profile

{'name': 'Qwen3 32B',
 'release_date': '2024-12-23',
 'last_updated': '2024-12-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 40960,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

## DataClasses
A dataclass is a class typically containing mainly data, although there aren't really any restrications. You create it using the @dataclass decorater.

In [18]:
import os
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")


In [ ]:

resultfrom pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person"""
    name:str = Field(description="The name of the person")
    email:str = Field(description="The email address of the person")
    phone:str = Field( description="The phone number of the person")
    
agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo # auto-selects ProvidersStrategy
)

result= agent.invoke({
    "messages":[{"role":"user","content":"Extract contact info from: John Doe, john@example.com, (551) 123-3344"}]
    
})


{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (551) 123-3344', additional_kwargs={}, response_metadata={}, id='1687642e-966f-4186-81d3-c03bf9303e9a'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(551) 123-3344"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 548, 'prompt_tokens': 204, 'total_tokens': 752, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 512, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dw9gzBnIUewmQGk1fKCx4aIBnMKPC', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f1456-a869-7980-b15e-56a6a1ed652c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 204, 'output

In [24]:
print(result["structured_response"])

name='John Doe' email='john@example.com' phone='(551) 123-3344'


## TypedDict

In [27]:
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for a person"""
    name: str   # The name of the person
    email: str  # The email address of the person
    phone: str  # The phone number of the person
    
agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo # auto-selects ProvidersStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (551) 123-3344"}]
})

result["structured_response"]

ImportError: cannot import name 'TypedDict' from 'pydantic' (/Users/xe/Documents/LANGCHAIN_PROJECT/langchainupdated/.venv/lib/python3.13/site-packages/pydantic/__init__.py)